1 Zuerst werden die Daten ordentlich geladen

In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/raw")

measures = pd.read_csv(DATA_DIR / "measures.csv", sep=";", decimal=",")
to_predict = pd.read_csv(DATA_DIR / "to_predict.csv", sep=";", decimal=",")

print("Measures:", measures.shape)
print("To predict:", to_predict.shape)

measures.head()

Measures: (7352, 563)
To predict: (2947, 562)


,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",subject,activity
0,0.288585,-0.020294,-0.132905,-0.995279,-0.983111,-0.913526,-0.995112,-0.983185,-0.923527,-0.934724,...,-0.710304,-0.112754,0.030400,-0.464761,-0.018446,-0.841247,0.179941,-0.058627,1.0,STANDING
1,0.278419,-0.016411,-0.123520,-0.998245,-0.975300,-0.960322,-0.998807,-0.974914,-0.957686,-0.943068,...,-0.861499,0.053477,-0.007435,-0.732626,0.703511,-0.844788,0.180289,-0.054317,1.0,STANDING
2,0.279653,-0.019467,-0.113462,-0.995380,-0.967187,-0.978944,-0.996520,-0.963668,-0.977469,-0.938692,...,-0.760104,-0.118559,0.177899,0.100699,0.808529,-0.848933,0.180637,-0.049118,1.0,STANDING
3,0.279174,-0.026201,-0.123283,-0.996091,-0.983403,-0.990675,-0.997099,-0.982750,-0.989302,-0.938692,...,-0.482845,-0.036788,-0.012892,0.640011,-0.485366,-0.848649,0.181935,-0.047663,1.0,STANDING
4,0.276629,-0.016570,-0.115362,-0.998139,-0.980817,-0.990482,-0.998321,-0.979672,-0.990441,-0.942469,...,-0.699205,0.123320,0.122542,0.693578,-0.615971,-0.847865,0.185151,-0.043892,1.0,STANDING


2 Es werden nach Missing Values gesucht

In [2]:
print("Anzahl fehlender Werte in measures:", measures.isnull().sum().sum())
print("Anzahl Spalten mit fehlenden Werten in measures:", (measures.isna().sum() > 0).sum())

print("Anzahl fehlender Werte in to_predict:", to_predict.isnull().sum().sum())
print("Anzahl Spalten mit fehlenden Werten in to_predict:", (to_predict.isna().sum() > 0).sum())

Anzahl fehlender Werte in measures: 0
Anzahl Spalten mit fehlenden Werten in measures: 0
Anzahl fehlender Werte in to_predict: 0
Anzahl Spalten mit fehlenden Werten in to_predict: 0


3 Klassenverteilung und Subject prüfen:

- Wir müssen vor dem Modellieren verstehen, wie die Zielvariable und die Personen im Datensatz verteilt sind.
- Bei Klassenverteilung: Gibt es unausgeglichene Klassen?
- Bei Subject: Welche Personen sind überhaupt vorhanden?
- Beobachtung pro Subject: Ist der Testsplit sinnvoll groß?

In [3]:
# Erste Bestandaufnahme der Daten
# Wie oft jede Aktivität in measures vorkommt
activity_counts = measures["activity"].value_counts()
print("Aktivitäten in measures:", activity_counts)

# In Relativer Häufigkeit:
print("\n")
print(measures["activity"].value_counts(normalize=True).round(3))

# Subjects
print("\n")
subjects = sorted(measures["subject"].unique())
subjects = [int(s) for s in subjects]
print("Subjects in measures:", subjects)

#Wie viele Messfenter pro Person
print("\n")
print("Anzahl Messfenster pro Person:", measures["subject"].value_counts().sort_index())

Aktivitäten in measures: activity
LAYING                1407
STANDING              1374
SITTING               1286
WALKING               1226
WALKING_UPSTAIRS      1073
WALKING_DOWNSTAIRS     986
Name: count, dtype: int64


activity
LAYING                0.191
STANDING              0.187
SITTING               0.175
WALKING               0.167
WALKING_UPSTAIRS      0.146
WALKING_DOWNSTAIRS    0.134
Name: proportion, dtype: float64


Subjects in measures: [1, 3, 5, 6, 7, 8, 11, 14, 15, 16, 17, 19, 21, 22, 23, 25, 26, 27, 28, 29, 30]


Anzahl Messfenster pro Person: subject
1.0     347
3.0     341
5.0     302
6.0     325
7.0     308
8.0     281
11.0    316
14.0    323
15.0    328
16.0    366
17.0    368
19.0    360
21.0    408
22.0    321
23.0    372
25.0    409
26.0    392
27.0    376
28.0    382
29.0    344
30.0    383
Name: count, dtype: int64
